# 02 · Chunking Strategy Comparison
Run once. Compares 4 strategies on a 200-paper sample. Winner is used in all scale tiers.

In [3]:
import os, re, json
import pandas as pd
import numpy as np

CORD_META = '../../1_data/raw/metadata.csv'
if not os.path.exists(CORD_META):
    # Try common Kaggle/local paths
    for p in ['../../1_data/metadata.csv', '../../../metadata.csv']:
        if os.path.exists(p):
            CORD_META = p
            break
assert os.path.exists(CORD_META), f'CORD-19 metadata not found at {CORD_META}'

SAMPLE_N    = 200
RANDOM_SEED = 42
RESULTS_DIR = '../../4_results/General'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'CORD-19 found: {CORD_META}')

CORD-19 found: ../../1_data/raw/metadata.csv


In [4]:
# Load sample
meta = pd.read_csv(CORD_META, low_memory=False)
meta = meta.dropna(subset=['abstract'])
meta = meta[meta['abstract'].str.len() > 100]
sample = meta.sample(SAMPLE_N, random_state=RANDOM_SEED)
texts = (sample['title'].fillna('') + ' ' + sample['abstract'].fillna('')).tolist()
print(f'Loaded {len(texts)} texts for comparison')
print(f'Avg text length: {sum(len(t.split()) for t in texts)/len(texts):.0f} words')

Loaded 200 texts for comparison
Avg text length: 225 words


In [5]:
# ── 4 chunking strategies ──────────────────────────────────────────────

def fixed_token(text, size=400, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(' '.join(words[i:i+size]))
        i += size - overlap
    return chunks

def sentence_window(text, window=5, stride=3):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    sents = [s for s in sents if len(s.split()) > 3]
    chunks = []
    for i in range(0, max(1, len(sents) - window + 1), stride):
        chunks.append(' '.join(sents[i:i+window]))
    return chunks or [text]

def sliding_window(text, size=300, overlap=100):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(' '.join(words[i:i+size]))
        i += size - overlap
    return chunks

def paragraph_split(text):
    paras = [p.strip() for p in re.split(r'\n{2,}|(?<=[.])  +', text) if len(p.split()) > 10]
    return paras or [text]

STRATEGIES = {
    'fixed_token_400_50':    lambda t: fixed_token(t, 400, 50),
    'sentence_window_5_3':   lambda t: sentence_window(t, 5, 3),
    'sliding_window_300_100':lambda t: sliding_window(t, 300, 100),
    'paragraph_split':       lambda t: paragraph_split(t),
}
print('Strategies defined:', list(STRATEGIES.keys()))

Strategies defined: ['fixed_token_400_50', 'sentence_window_5_3', 'sliding_window_300_100', 'paragraph_split']


In [6]:
# ── Evaluate each strategy ─────────────────────────────────────────────
def ends_mid_sentence(chunk):
    """True if the chunk ends abruptly (no terminal punctuation)."""
    return not re.search(r'[.!?]\s*$', chunk.strip())

def starts_mid_sentence(chunk):
    """True if chunk starts with lowercase (likely mid-sentence)."""
    first = chunk.strip()
    return bool(first) and first[0].islower()

results = []
for name, fn in STRATEGIES.items():
    all_chunks = []
    for text in texts:
        all_chunks.extend(fn(text))

    lengths = [len(c.split()) for c in all_chunks]
    bad_end   = sum(ends_mid_sentence(c) for c in all_chunks)
    bad_start = sum(starts_mid_sentence(c) for c in all_chunks)
    clean_both = sum(not ends_mid_sentence(c) and not starts_mid_sentence(c)
                     for c in all_chunks)

    results.append({
        'strategy':        name,
        'total_chunks':    len(all_chunks),
        'avg_len_words':   round(np.mean(lengths), 1),
        'std_len_words':   round(np.std(lengths), 1),
        'min_len':         min(lengths),
        'max_len':         max(lengths),
        'bad_end_pct':     round(100 * bad_end / len(all_chunks), 1),
        'bad_start_pct':   round(100 * bad_start / len(all_chunks), 1),
        'clean_both_pct':  round(100 * clean_both / len(all_chunks), 1),
    })
    print(f'{name}: {len(all_chunks)} chunks, avg {np.mean(lengths):.0f} words, '
          f'clean={100*clean_both/len(all_chunks):.1f}%')

df = pd.DataFrame(results)
df.to_csv(f'{RESULTS_DIR}/chunking_boundary_quality.csv', index=False)
print(f'\nSaved → {RESULTS_DIR}/chunking_boundary_quality.csv')

fixed_token_400_50: 212 chunks, avg 213 words, clean=85.4%
sentence_window_5_3: 434 chunks, avg 117 words, clean=95.2%
sliding_window_300_100: 324 chunks, avg 163 words, clean=52.8%
paragraph_split: 200 chunks, avg 225 words, clean=91.0%

Saved → ../../4_results/General/chunking_boundary_quality.csv


In [7]:
print('\n=== CHUNKING STRATEGY COMPARISON ===')
print(df[['strategy','total_chunks','avg_len_words','clean_both_pct']].to_string(index=False))

winner = df.loc[df['clean_both_pct'].idxmax(), 'strategy']
print(f'\nWinner (highest clean_both_pct): {winner}')
print('\nDecision for all scale tiers: fixed_token_400_50')
print('  Rationale: predictable chunk size for embedding batches;')
print('  50-token overlap preserves cross-boundary context;')
print('  consistent with BGE-M3 512-token input limit.')


=== CHUNKING STRATEGY COMPARISON ===
              strategy  total_chunks  avg_len_words  clean_both_pct
    fixed_token_400_50           212          213.4            85.4
   sentence_window_5_3           434          116.9            95.2
sliding_window_300_100           324          162.8            52.8
       paragraph_split           200          224.6            91.0

Winner (highest clean_both_pct): sentence_window_5_3

Decision for all scale tiers: fixed_token_400_50
  Rationale: predictable chunk size for embedding batches;
  50-token overlap preserves cross-boundary context;
  consistent with BGE-M3 512-token input limit.
